# Performance analytics

**Prerequisites:** complete `01_foundations/core_types_and_money.ipynb`, `01_foundations/dates_calendars_schedules.ipynb`, `01_foundations/market_data_and_curves.ipynb`, and `01_foundations/math_toolkit.ipynb` in this curriculum (foundations track: core types, dates/calendars, market data/curves, and the math toolkit). This notebook assumes you are comfortable importing `finstack_quant` and working with simple Python lists.

**In this notebook:** return-based performance measurement — standalone metrics on `list[float]` inputs and the panel-level `Performance` engine (pandas-backed) for multi-asset summaries, correlations, rolling risk ratios, and drawdown episodes.

## Performance measurement in a nutshell

**Returns** are the basic input: from a price series you compute simple period returns, then compound them, measure dispersion (volatility), and compare reward to risk (Sharpe, Sortino, Calmar).

**Drawdowns** track peak-to-trough loss along a cumulative path; duration and depth matter as much as the headline max drawdown.

**Tail risk** (VaR and expected shortfall) summarizes how bad the left tail of the return distribution is — useful for risk limits and stress-style intuition.

Finstack Quant exposes fast Rust-backed implementations in `finstack_quant.analytics`: lightweight **standalone functions** for vectors, and a **`Performance`** class when you have a dated panel (multiple tickers, shared calendar).

### Standalone functions on return (and price) lists

Construct `Performance` from a price (or return) panel. Every metric — return/risk scalars, drawdowns, periodic returns, benchmark alpha/beta, basic factor models — is reachable as a method on the resulting instance.

Below: compounding, mean return, volatility (raw and annualized), classic ratios, drawdowns from returns, and a few distribution / streak metrics. Every result is printed so you can scan the magnitudes.

CAGR annualized from very short windows is unreliable

In [ ]:
from datetime import date, timedelta

from finstack_quant.analytics import Performance

prices = [100.0, 105.0, 103.0, 108.0, 107.0, 112.0, 110.0, 115.0]
dates = [date(2024, 1, 1) + timedelta(days=i) for i in range(len(prices))]
perf = Performance.from_arrays(dates, [prices], ["SAMPLE"])

print("Active dates count:", len(perf.dates()))
print("CAGR:", round(perf.cagr()[0], 6))
print("Mean return (ann.):", round(perf.mean_return()[0], 6))
print("Volatility (per-period):", round(perf.volatility(annualize=False)[0], 6))
print("Volatility (ann.):", round(perf.volatility(annualize=True)[0], 6))
print("Sharpe:", round(perf.sharpe()[0], 6))
print("Sortino:", round(perf.sortino()[0], 6))
print("Max drawdown:", round(perf.max_drawdown()[0], 6))
print("Calmar:", round(perf.calmar()[0], 6))
print("Omega ratio:", round(perf.omega_ratio()[0], 6))
print("Gain-to-pain:", round(perf.gain_to_pain()[0], 6))
print("Downside deviation:", round(perf.downside_deviation()[0], 6))
print("Skewness:", round(perf.skewness()[0], 6))
print("Kurtosis:", round(perf.kurtosis()[0], 6))
print("Geometric mean:", round(perf.geometric_mean()[0], 6))

### Additional ratios (treynor, m-squared, sterling, pain, ulcer, recovery, skew/kurt combined)

A few more widely-used scalars not shown in the first pass.


In [ ]:
print("Treynor:", round(perf.treynor()[0], 6))
print("M-squared:", round(perf.m_squared()[0], 6))
print("Sterling (n=3):", round(perf.sterling_ratio(n=3)[0], 6))
print("Pain ratio:", round(perf.pain_ratio()[0], 6))
print("Ulcer index:", round(perf.ulcer_index()[0], 6))
print("Recovery factor:", round(perf.recovery_factor()[0], 6))
sk, ku = perf.skew_kurt()
print("Skew/Kurt (tuple):", round(sk[0], 4), round(ku[0], 4))


### VaR, expected shortfall, and parametric variants

**Historical VaR** uses the empirical quantile of returns. In this library, `confidence` is the *VaR confidence level* in \((0, 1)\): for example `0.95` means **95% VaR**, i.e. the loss threshold at the **5th percentile** of the return distribution (see the Rust docstring for `value_at_risk`).

**Expected shortfall** (CVaR) averages returns worse than that threshold. **Parametric** and **Cornish–Fisher** variants adjust for Gaussian or skew/kurtosis structure.

All of the following print on the same `rets` sample as above.

In [ ]:
print("Historical VaR (95% confidence):", round(perf.value_at_risk(0.95)[0], 6))
print("Expected shortfall (95%):", round(perf.expected_shortfall(0.95)[0], 6))
print("Parametric VaR (95%):", round(perf.parametric_var(0.95)[0], 6))
print("Cornish–Fisher VaR (95%):", round(perf.cornish_fisher_var(0.95)[0], 6))

### `Performance`: dated panel analytics

`Performance` is constructed from a **pandas** `DataFrame` of prices (or via `Performance.from_arrays` with parallel lists). It aligns tickers on a calendar and returns **per-ticker vectors** for most metrics (`cagr()`, `sharpe()`, …).

Use `rolling_sharpe(ticker_idx, ...)`, `rolling_volatility(...)`, and `drawdown_details(ticker_idx, n=...)` when you need **time series** or **episode lists**. `period_stats(ticker_idx, agg_freq=...)` summarizes winning/losing streaks at a chosen aggregation.

In [ ]:
from datetime import date, timedelta

from finstack_quant.analytics import Performance

dates = [date(2024, 1, 1) + timedelta(days=i) for i in range(252)]
spy_prices = [100.0]
for i in range(251):
    spy_prices.append(spy_prices[-1] * (1.0 + 0.0004 + (i % 17) * 1e-5))

perf = Performance.from_arrays(dates, [spy_prices], ["SPY"])

print("Tickers:", perf.ticker_names)
print("Freq:", perf.freq)
print("Dates count:", len(perf.dates()))

cagr_v = perf.cagr()
print("CAGR:", [round(x, 6) for x in cagr_v])
print("Mean return (ann.):", [round(x, 6) for x in perf.mean_return()])
print("Sharpe:", [round(x, 6) for x in perf.sharpe()])
print("Sortino:", [round(x, 6) for x in perf.sortino()])
print("Max drawdown:", [round(x, 6) for x in perf.max_drawdown()])
print("Max DD duration (days):", perf.max_drawdown_duration())
print("Volatility (ann.):", [round(x, 6) for x in perf.volatility()])
print("Calmar:", [round(x, 6) for x in perf.calmar()])

### Rolling metrics, drawdown paths, and period stats

`drawdown_series()` returns a drawdown path per ticker. `drawdown_details` lists the worst episodes with start, valley, and (if recovered) end dates.

Rolling helpers return objects with `.values` and `.dates()` for plotting or tables; here we print compact summaries.

In [ ]:
dd_series = perf.drawdown_series()
print("First ticker drawdown series (first 5):", [round(x, 6) for x in dd_series[0][:5]])

episodes = perf.drawdown_details(0, n=3)
print("Top drawdown episodes (SPY):")
for i, ep in enumerate(episodes, start=1):
    end = ep.end
    print(
        f"  {i}) start={ep.start} valley={ep.valley} end={end!s} "
        f"days={ep.duration_days} max_dd={round(ep.max_drawdown, 6)}"
    )

rs = perf.rolling_sharpe(0, window=63, risk_free_rate=0.0)
rv = perf.rolling_volatility(0, window=63)
print("Rolling Sharpe: n=", len(rs.values), "first3=", [round(x, 6) for x in rs.values[:3]])
print("Rolling vol: n=", len(rv.values), "last3=", [round(x, 6) for x in rv.values[-3:]])

ps = perf.period_stats(0, agg_freq="monthly")
print(
    "Monthly period stats — win_rate:",
    round(ps.win_rate, 4),
    "best:",
    round(ps.best, 6),
    "worst:",
    round(ps.worst, 6),
    "consecutive_wins:",
    ps.consecutive_wins,
    "consecutive_losses:",
    ps.consecutive_losses,
)

## Mini-example: three-ticker synthetic panel

We simulate **252 daily** prices for **SPY**, **QQQ**, and **BND** with a simple Gaussian random walk: higher drift/vol for equities, muted dynamics for bonds. The seed is fixed so the notebook is reproducible.

We then rank key ratios per ticker, show the **correlation matrix**, summarize **rolling Sharpe** for SPY, and print **drawdown episodes** for each name.

In [ ]:
import random
from datetime import date, timedelta

from finstack_quant.analytics import Performance

random.seed(42)

n = 252
dates = [date(2024, 1, 1) + timedelta(days=i) for i in range(n)]


def random_walk_prices(start: float, daily_drift: float, daily_vol: float, steps: int) -> list[float]:
    out = [start]
    for _ in range(steps - 1):
        shock = random.gauss(0.0, daily_vol)
        out.append(out[-1] * (1.0 + daily_drift + shock))
    return out


spy_px = random_walk_prices(100.0, 0.00035, 0.008, n)
qqq_px = random_walk_prices(100.0, 0.00055, 0.012, n)
bnd_px = random_walk_prices(100.0, 0.00008, 0.0015, n)

tickers = ["SPY", "QQQ", "BND"]
panel = Performance.from_arrays(dates, [spy_px, qqq_px, bnd_px], tickers)

cagr_ = panel.cagr()
sharpe_ = panel.sharpe()
sortino_ = panel.sortino()
mdd_ = panel.max_drawdown()

print("Per-ticker headline metrics:")
for name, cg, sh, so, dd in zip(tickers, cagr_, sharpe_, sortino_, mdd_):
    print(
        f"  {name}: CAGR={round(cg, 6)} Sharpe={round(sh, 6)} Sortino={round(so, 6)} maxDD={round(dd, 6)}"
    )

corr = panel.correlation_matrix()
print("Correlation matrix (rows/cols =", tickers, "):")
for row in corr:
    print("  ", [round(x, 4) for x in row])

rs_spy = panel.rolling_sharpe(0, window=63)
print(
    "SPY rolling Sharpe: len=",
    len(rs_spy.values),
    "sample tail=",
    [round(x, 6) for x in rs_spy.values[-3:]],
)

for idx, name in enumerate(tickers):
    dds = panel.drawdown_details(idx, n=2)
    print(f"Drawdown details ({name}), top {len(dds)} episodes:")
    for j, ep in enumerate(dds, start=1):
        print(
            f"  {j}) max_dd={round(ep.max_drawdown, 6)} start={ep.start} valley={ep.valley} end={ep.end!s}"
        )

## Alternative constructors and DataFrame exports

`Performance` supports multiple construction paths:

- `Performance(prices_df)` — direct from a pandas DataFrame of prices (date index, one column per ticker).
- `Performance.from_returns(returns_df)` — from a returns DataFrame.
- `from_arrays` / `from_returns_arrays` — from raw lists (shown above).

Export helpers turn the engine into tables for downstream use:

- `summary_to_dataframe()` — scalar metrics for all tickers as a DataFrame.
- `cumulative_returns_to_dataframe()` / `periodic_returns_to_dataframe(freq)` — time series and bucketed returns.


In [ ]:
import pandas as pd

# Build a tiny price panel as a DataFrame (same shape as the 3-ticker example above)
dates = [date(2024, 1, 1) + timedelta(days=i) for i in range(10)]
px = {
    "SPY": [100.0 + i*0.1 for i in range(10)],
    "BND": [100.0 - i*0.02 for i in range(10)],
}
prices_df = pd.DataFrame(px, index=dates)
prices_df.index.name = "date"

perf_df = Performance(prices_df, benchmark_ticker="SPY", freq="daily")
print("Direct DF ctor tickers:", perf_df.ticker_names)
print("Direct DF ctor CAGR:", [round(x, 6) for x in perf_df.cagr()])

# Side-by-side: from_arrays (earlier), direct DF ctor (above), and from_returns (here)
# from_returns path (compute simple returns from the same prices)
ret_df = prices_df.pct_change().dropna()
perf_ret = Performance.from_returns(ret_df, benchmark_ticker="SPY", freq="daily")
print("from_returns tickers:", perf_ret.ticker_names)
print("from_returns Sharpe:", [round(x, 6) for x in perf_ret.sharpe()])

# DataFrame exports
print("\nSummary (head):")
print(perf_df.summary_to_dataframe().to_string())

print("\nCumulative returns (first 3 rows):")
print(perf_df.cumulative_returns_to_dataframe().head(3).to_string())

## Windowing, resets, and structured result access

Use `reset_date_range` / `reset_bench_ticker` to narrow the analysis window or change the benchmark without rebuilding. Result objects carry rich accessors (e.g. `LookbackReturns.fytd`).


In [ ]:
# recreate a slightly longer panel for reset demo
dates = [date(2024, 1, 1) + timedelta(days=i) for i in range(30)]
px = [100.0 + i*0.2 for i in range(30)]
p = Performance.from_arrays(dates, [px], ["TICK"])
print("full active dates:", len(p.active_dates()))
p.reset_date_range(dates[5], dates[-3])
print("after reset_date_range:", len(p.active_dates()))

# lookback with FYTD
lb = p.lookback_returns(dates[-1], fiscal_year_start_month=1)
print("lookback mtd/qtd/ytd sample:", round(lb.mtd[0],6), round(lb.qtd[0],6), round(lb.ytd[0],6))
print("fytd present:", lb.fytd is not None)

# one excess / outperformance series
ex = p.excess_returns([0.0]*len(p.dates()))
print("excess series len:", len(ex[0]))


### Named lookback result

`lookback_returns` returns a `LookbackReturns` object, so code can access MTD/QTD/YTD/FYTD fields by name.

In [ ]:
from finstack_quant.analytics import LookbackReturns

print("LookbackReturns type:", LookbackReturns.__name__)
print("lb is LookbackReturns:", isinstance(lb, LookbackReturns))
print("YTD sample:", round(lb.ytd[0], 6))

## Structured result types

`Performance`'s structured methods return typed value objects rather than loose tuples: `period_stats` → `PeriodStats`, `beta` → `BetaResult`, `greeks` → `GreeksResult`, `rolling_greeks` → `RollingGreeks`, `rolling_volatility`/`rolling_sharpe`/… → `DatedSeries`, `drawdown_details` → `DrawdownEpisode`, and `multi_factor_greeks` → `MultiFactorResult`.

In [ ]:
import random as _random
from datetime import date as _date, timedelta as _timedelta

from finstack_quant.analytics import (
    PeriodStats,
    BetaResult,
    GreeksResult,
    RollingGreeks,
    MultiFactorResult,
    DrawdownEpisode,
    DatedSeries,
)

_random.seed(1)
_n = 180
_dates = [_date(2024, 1, 1) + _timedelta(days=i) for i in range(_n)]

def _walk(drift: float) -> list[float]:
    px = [100.0]
    for _ in range(_n - 1):
        px.append(px[-1] * (1.0 + drift + _random.gauss(0, 0.01)))
    return px

# Final ticker serves as the benchmark for beta/greeks
sp = Performance.from_arrays(_dates, [_walk(0.0004), _walk(0.0002), _walk(0.0003)], ["A", "B", "BENCH"])

ps = sp.period_stats(0)
betas = sp.beta()
gk = sp.greeks()
rg = sp.rolling_greeks(0, window=30)
rv = sp.rolling_volatility(0, window=30)
dd = sp.drawdown_details(0, n=3)
mf = sp.multi_factor_greeks(0, [[_random.gauss(0, 0.01) for _ in range(_n - 1)]])

print("period_stats           ->", type(ps).__name__, "| best:", round(ps.best, 4))
print("beta[0]                ->", type(betas[0]).__name__, "| beta:", round(betas[0].beta, 4))
print("greeks[0]              ->", type(gk[0]).__name__, "| alpha:", round(gk[0].alpha, 6))
print("rolling_greeks         ->", type(rg).__name__, "| points:", len(rg.betas))
print("rolling_volatility     ->", type(rv).__name__, "| col:", rv.value_column)
print("drawdown_details[0]    ->", type(dd[0]).__name__, "| max_dd:", round(dd[0].max_drawdown, 4))
print("multi_factor_greeks    ->", type(mf).__name__, "| r2:", round(mf.r_squared, 4))

## Takeaways

- **Performance is the single entry point:** Construct it once from prices (or returns) and access scalars (`volatility`, `sortino`, `max_drawdown`), tail metrics (`value_at_risk`, `expected_shortfall`), and panel views (`correlation_matrix`) as methods on the same instance.
- **Confidence, not alpha:** pass `confidence=0.95` for 95% historical VaR in this API (the implementation targets the \((1 - \text{confidence})\) quantile).
- **Rolling and episode views:** `rolling_sharpe`, `rolling_volatility`, `rolling_greeks`, and `drawdown_details` provide time-series and episode-level forensics directly on the same `Performance` instance.
- **Reproducibility:** fix random seeds when you simulate; keep calendars consistent (`daily` frequency) when annualizing or comparing to 252-day conventions.

Next steps: combine these metrics with real curves and calendars from the foundations notebooks, or wire returns from your own portfolio accounting pipeline.